## 00. load package

In [24]:

import intake
from typing import Tuple, List, Optional, Union, Dict
import scipy.signal as signal
from global_land_mask import globe
import os
import pickle
import sys
from pathlib import Path
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools.utils import dataarray_to_equatorial_latlon_grid,get_region_healpix_,create_cmap_from_string,dataarray_healpix_to_equatorial_latlon
print("="*70)
print("✅ 模块重新加载完成")
print("="*70)
import mpi4py
import logging
import glob
import psutil
import xarray as xr
import numpy as np
# ============= 新增：Dask 配置优化 =============
import dask
from dask.diagnostics import ProgressBar

# 配置 Dask 以避免内存溢出
dask.config.set({
    'array.slicing.split_large_chunks': True,
    'distributed.worker.memory.target': 0.95,  # 75% 内存使用上限
    'distributed.worker.memory.spill': 0.90,   # 85% 时开始写入磁盘
    'distributed.worker.memory.pause': 0.99,   # 95% 时暂停
    'array.chunk-size': '128MiB'  # 设置合理的分块大小
})

print("✅ Dask 内存优化配置完成")
print("="*70)


✅ 模块重新加载完成
✅ Dask 内存优化配置完成


In [25]:
# ============= 增强版：内存监控和诊断工具 =============


def get_memory_usage():
    """获取当前内存使用情况"""
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    mem_gb = mem_info.rss / 1024**3  # 转换为 GB
    
    # 系统总内存
    virtual_mem = psutil.virtual_memory()
    total_gb = virtual_mem.total / 1024**3
    available_gb = virtual_mem.available / 1024**3
    percent_used = virtual_mem.percent
    
    return {
        'process_gb': mem_gb,
        'system_total_gb': total_gb,
        'system_available_gb': available_gb,
        'system_percent': percent_used
    }

def print_memory_status(label=""):
    """打印内存状态"""
    mem = get_memory_usage()
    print(f"💾 内存状态 {label}")
    print(f"   进程使用: {mem['process_gb']:.2f} GB")
    print(f"   系统可用: {mem['system_available_gb']:.2f} GB / {mem['system_total_gb']:.2f} GB")
    print(f"   系统使用率: {mem['system_percent']:.1f}%")
    
    # 警告阈值
    if mem['system_percent'] > 85:
        print(f"   ⚠️ 警告: 内存使用率过高 ({mem['system_percent']:.1f}%)!")
        return False
    return True

# 测试内存监控
print("="*70)
print("🔧 内存监控工具已加载")
print_memory_status("(初始状态)")
print("="*70)

🔧 内存监控工具已加载
💾 内存状态 (初始状态)
   进程使用: 16.04 GB
   系统可用: 325.53 GB / 501.87 GB
   系统使用率: 35.1%


In [26]:
# 网格转换参数

def dataarray_to_equatorial_latlon_grid(
    dataarray: xr.DataArray, grid_type: str, grid_dict: Optional[dict]
) -> xr.DataArray:
    """转换数据到赤道经纬度网格"""
    if grid_type == "latlon":
        return dataarray
    elif grid_type == "healpix":
        if grid_dict is None:
            raise ValueError("No grid_dict provided for healpix conversion.")
        
        # 检查是否有时间维度
        has_time = 'time' in dataarray.dims
        
        if not has_time:
            # 如果没有时间维度，添加一个虚拟的时间维度
            dataarray_with_time = dataarray.expand_dims(time=[np.datetime64('1980-01-01')])
            result = dataarray_healpix_to_equatorial_latlon(dataarray_with_time, **grid_dict)
            # 移除虚拟的时间维度
            result = result.squeeze('time', drop=True)
            return result
        else:
            return dataarray_healpix_to_equatorial_latlon(dataarray, **grid_dict)
    else:
        raise ValueError("Grid type not found.")




In [27]:
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
# 创建数据保存目录
DATA_DIR = "/work/mh1498/m301257/processed_data_lat_30"
os.makedirs(DATA_DIR, exist_ok=True)
ds = (cat.ICON.C5.AMIP_CNTL.to_dask()).sel(time=slice("1980", "1993"))
ds 


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


<xarray.Dataset> Size: 6TB
Dimensions:             (time: 5114, cell: 786432, level_full: 26,
                         level_half: 26)
Coordinates:
  * time                (time) datetime64[ns] 41kB 1980-01-01 ... 1993-12-31
  * level_full          (level_full) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
  * level_half          (level_half) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
    healpix             int64 8B 1
Dimensions without coordinates: cell
Data variables: (12/47)
    clivi               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    cllvi               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hus2m               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hfls                (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hfss                (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    pr                  (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    ...                  ...
    phalf               (time, level_half, cell) float32 418GB dask.array<chunksize=(32, 4, 16384), meta=np.ndarray>
    cell_elevation      (cell) float64 6MB dask.array<chunksize=(262144,), meta=np.ndarray>
    cell_sea_land_mask  (cell) int32 3MB dask.array<chunksize=(262144,), meta=np.ndarray>
    zg                  (level_full, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
    zghalf              (level_half, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
    dzghalf             (level_full, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
Attributes:
    CDI:                       Climate Data Interface version 2.4.0 (https://...
    Conventions:               CF-1.6
    source:                    https://gitlab.dkrz.de/icon/icon-mpim.git@6684...
    institution:               Max Planck Institute for Meteorology/Deutscher...
    title:                     ICON simulation
    references:                see MPIM/DWD publications
    comment:                   Lukas Kluft (kluftluka) on nid006406 (Linux 5....
    cdo_bitrounding_numbits:   13
    CDO:                       Climate Data Operators version 2.4.0 (https://...
    cdo_openmp_thread_number:  4
    history:                   Wed Mar 13 20:19:59 2024: ncrename -d cells,ce...
    NCO:                       netCDF Operators version 5.0.1 (Homepage = htt...

In [28]:
# def process_var_data(var_name, experiment_name, dataset_key, save_dir, grid_dict, target_lat, target_lon, 
#                      has_level=True, level_slice=(30, None), has_time=True):
#     """
#     处理变量数据（支持3D和2D），转换网格并插值后保存
    
#     Parameters:
#     -----------
#     var_name : str
#         变量名 ('wa', 'hus', 'ta', 'pr', 'phalf', 'pfull', 'zg')
#     experiment_name : str
#         实验名称（用于显示）
#     dataset_key : str
#         在catalog中的数据集键名
#     save_dir : str
#         保存目录
#     grid_dict : dict
#         网格转换参数
#     target_lat : array
#         目标纬度
#     target_lon : array
#         目标经度
#     has_level : bool
#         是否为3D数据（有level维度）。True=3D，False=2D
#     level_slice : tuple
#         level切片范围，仅当has_level=True时有效
#     has_time : bool
#         是否有时间维度。False时跳过时间切片（例如zg变量）
#     """
#     import time
#     import gc  # 垃圾回收
    
#     if not has_time:
#         data_type = "静态场 (无时间维度)"
#     elif has_level:
#         data_type = "3D (多层级)"
#     else:
#         data_type = "2D (时间序列)"
#     print("="*70)
#     print(f"🔄 处理 {var_name.upper()} - {experiment_name} [{data_type}]")
#     print("="*70)
    
#     # 创建变量和实验的子目录
#     if has_level:
#         exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}_layers")
#     else:
#         exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}")
#     os.makedirs(exp_save_dir, exist_ok=True)
#     print(f"📁 保存路径: {exp_save_dir}")
    
#     # 加载数据（只读取元数据）
#     print(f"📖 读取数据元信息...")
    
#     # 自动识别 level 维度名称
#     level_dim_name = None
    
#     try:
#         if has_level:
#             # 先加载变量以检查其维度
#             if has_time:
#                 var_temp = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(time=slice("1980", "1993"))
#             else:
#                 # 无时间维度的变量（如zg）
#                 var_temp = cat.ICON.C5[dataset_key].to_dask()[var_name]
            
#             # 自动检测 level 维度名称
#             possible_level_dims = ['level_full', 'level_half', 'level']
#             for dim in possible_level_dims:
#                 if dim in var_temp.dims:
#                     level_dim_name = dim
#                     break
            
#             if level_dim_name is None:
#                 raise ValueError(f"无法找到 level 维度。变量 {var_name} 的维度: {list(var_temp.dims)}")
            
#             print(f"✅ 自动检测到 level 维度: {level_dim_name}")
            
#             # 根据检测到的维度名称选择数据
#             var_full = var_temp.sel({level_dim_name: slice(*level_slice)})
#             levels = var_full[level_dim_name].values
#             n_levels = len(levels)
            
#             print(f"✅ 数据信息:")
#             print(f"   变量: {var_name}")
#             if has_time:
#                 print(f"   时间范围: 1980-1993")
#                 print(f"   时间步数: {len(var_full.time)}")
#             else:
#                 print(f"   静态场 (无时间维度)")
#             print(f"   总层数: {n_levels}")
#             print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
#         else:
#             # 2D数据：只有时间维度
#             if has_time:
#                 var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(
#                     time=slice("1980", "1993")
#                 )
                
#                 print(f"✅ 数据信息:")
#                 print(f"   变量: {var_name}")
#                 print(f"   时间范围: 1980-1993")
#                 print(f"   时间步数: {len(var_full.time)}")
#             else:
#                 raise ValueError(f"2D数据必须有时间维度，但 {var_name} 没有时间维度")
#     except Exception as e:
#         print(f"❌ 数据加载失败: {str(e)}")
#         return
    
#     print("="*70)
    
#     # 逐层处理（3D）或整体处理（2D）
#     total_start_time = time.time()
    
#     if has_level:
#         # ========== 3D数据：逐层处理 ==========
#         for idx, level in enumerate(levels, 1):
#             layer_start_time = time.time()
            
#             # 构建保存路径
#             save_path = os.path.join(exp_save_dir, f"{var_name}_lev_{int(level):03d}.nc")
            
#             # 检查是否已处理
#             if os.path.exists(save_path):
#                 print(f"✅ [{idx}/{n_levels}] Level {int(level):3d} - 已存在，跳过")
#                 continue
            
#             print(f"🔄 [{idx}/{n_levels}] 处理 Level {int(level):3d}...")
            
#             try:
#                 # 1. 选择单层数据（使用自动检测的维度名称）
#                 var_layer = var_full.sel({level_dim_name: level})
#                 print(f"   ├─ 选择层级完成 (使用 {level_dim_name})")
                
#                 # 2. 转换到经纬度网格
#                 var_lonlat = dataarray_to_equatorial_latlon_grid(var_layer, 'healpix', grid_dict)
#                 print(f"   ├─ 网格转换完成: {var_lonlat.shape}")
                
#                 # 3. 插值到2°x2°
#                 var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
#                 print(f"   ├─ 插值完成: {var_2deg.shape}")
                
#                 # 4. 转换为Dataset并设置变量名，然后保存
#                 ds_to_save = var_2deg.to_dataset(name=var_name)
                
#                 # 使用 compute() 触发计算并保存
#                 with ProgressBar():
#                     ds_to_save.to_netcdf(save_path, compute=True)
                
#                 # 清理内存
#                 del var_layer, var_lonlat, var_2deg, ds_to_save
#                 gc.collect()
                
               
                
#             except MemoryError:
#                 print(f"   ❌ 内存不足，跳过此层")
#                 gc.collect()
#                 continue
#             except Exception as e:
#                 print(f"   ❌ 处理失败: {str(e)}")
#                 gc.collect()
#                 continue
    
#     else:
#         # ========== 2D数据或静态场：处理 ==========
#         save_path = os.path.join(exp_save_dir, f"{var_name}_2deg_interp.nc")
        
#         # 检查是否已处理
#         if os.path.exists(save_path):
#             print(f"✅ 数据已存在，跳过处理")
#             print(f"   文件: {save_path}")
#         else:
#             print(f"🔄 开始处理{'静态场' if not has_time else '2D'}数据...")
            
#             try:
#                 if has_time:
#                     # 有时间维度：分批处理时间步以避免内存溢出
#                     n_times = len(var_full.time)
#                     batch_size = 365 * 2  # 每次处理2年数据
#                     n_batches = int(np.ceil(n_times / batch_size))
                    
#                     print(f"   ├─ 总时间步: {n_times}")
#                     print(f"   ├─ 批次大小: {batch_size}")
#                     print(f"   ├─ 总批次数: {n_batches}")
                    
#                     processed_data = []
                    
#                     for batch_idx in range(n_batches):
#                         start_idx = batch_idx * batch_size
#                         end_idx = min((batch_idx + 1) * batch_size, n_times)
                        
#                         print(f"   ├─ 批次 {batch_idx+1}/{n_batches}: 处理时间步 {start_idx}-{end_idx}...")
                        
#                         # 选择批次数据
#                         var_batch = var_full.isel(time=slice(start_idx, end_idx))
                        
#                         # 转换到经纬度网格
#                         var_lonlat = dataarray_to_equatorial_latlon_grid(var_batch, 'healpix', grid_dict)
                        
#                         # 插值到2°x2°
#                         var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                        
#                         # 立即计算并存储结果
#                         with ProgressBar():
#                             var_2deg_computed = var_2deg.compute()
                        
#                         processed_data.append(var_2deg_computed)
                        
#                         # 清理内存
#                         del var_batch, var_lonlat, var_2deg
#                         gc.collect()
                        
#                         print(f"   ├─ 批次 {batch_idx+1}/{n_batches} 完成")
                    
#                     # 合并所有批次
#                     print(f"   ├─ 合并所有批次...")
#                     var_final = xr.concat(processed_data, dim='time')
                    
#                     # 保存
#                     print(f"   ├─ 保存中...")
#                     ds_to_save = var_final.to_dataset(name=var_name)
#                     ds_to_save.to_netcdf(save_path)
                    
#                     # 清理内存
#                     del var_final, ds_to_save, processed_data
#                     gc.collect()
                    
#                 else:
#                     # 无时间维度（静态场，如zg）：直接处理
#                     print(f"   ├─ 静态场数据，直接处理...")
                    
#                     # 转换到经纬度网格
#                     var_lonlat = dataarray_to_equatorial_latlon_grid(var_full, 'healpix', grid_dict)
#                     print(f"   ├─ 网格转换完成")
                    
#                     # 插值到2°x2°
#                     var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
#                     print(f"   ├─ 插值完成")
                    
#                     # 计算并保存
#                     print(f"   ├─ 计算并保存中...")
#                     with ProgressBar():
#                         var_2deg_computed = var_2deg.compute()
                    
#                     ds_to_save = var_2deg_computed.to_dataset(name=var_name)
#                     ds_to_save.to_netcdf(save_path)
                    
#                     # 清理内存
#                     del var_lonlat, var_2deg, var_2deg_computed, ds_to_save
#                     gc.collect()
                
#                 total_time = time.time() - total_start_time
#                 print(f"   ✅ 保存完成: {os.path.basename(save_path)}")
#                 print(f"   ⏱️  总耗时: {total_time/60:.1f} 分钟")
                
#             except MemoryError:
#                 print(f"   ❌ 内存不足，尝试减小批次大小")
#                 gc.collect()
#                 return
#             except Exception as e:
#                 print(f"   ❌ 处理失败: {str(e)}")
#                 gc.collect()
#                 return
    
#     # 最后清理
#     gc.collect()


In [29]:
def process_var_data_safe(var_name, experiment_name, dataset_key, save_dir, grid_dict, 
                          target_lat, target_lon, has_level=True, level_slice=(30, None), 
                          has_time=True, max_retries=3, memory_threshold=80):
    """
    增强版数据处理函数 - 防崩溃版本
    
    新增参数:
    ---------
    max_retries : int
        每层最大重试次数
    memory_threshold : float
        内存使用率阈值（%），超过此值将跳过当前层
    """
    import time
    import gc
    
    # 打印初始内存状态
    print_memory_status("(处理开始前)")
    
    if not has_time:
        data_type = "静态场 (无时间维度)"
    elif has_level:
        data_type = "3D (多层级)"
    else:
        data_type = "2D (时间序列)"
    
    print("="*70)
    print(f"🔄 处理 {var_name.upper()} - {experiment_name} [{data_type}]")
    print("="*70)
    
    # 创建保存目录
    if has_level:
        exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}_layers")
    else:
        exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}")
    os.makedirs(exp_save_dir, exist_ok=True)
    print(f"📁 保存路径: {exp_save_dir}")
    
    # 进度跟踪文件
    progress_file = os.path.join(exp_save_dir, "_progress.txt")
    failed_levels_file = os.path.join(exp_save_dir, "_failed_levels.txt")
    
    print(f"📖 读取数据元信息...")
    
    level_dim_name = None
    
    try:
        if has_level:
            # 加载变量
            if has_time:
                var_temp = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(time=slice("1980", "1993"))
            else:
                var_temp = cat.ICON.C5[dataset_key].to_dask()[var_name]
            
            # 自动检测 level 维度
            possible_level_dims = ['level_full', 'level_half', 'level']
            for dim in possible_level_dims:
                if dim in var_temp.dims:
                    level_dim_name = dim
                    break
            
            if level_dim_name is None:
                raise ValueError(f"无法找到 level 维度。变量 {var_name} 的维度: {list(var_temp.dims)}")
            
            print(f"✅ 自动检测到 level 维度: {level_dim_name}")
            
            var_full = var_temp.sel({level_dim_name: slice(*level_slice)})
            levels = var_full[level_dim_name].values
            n_levels = len(levels)
            
            print(f"✅ 数据信息:")
            print(f"   变量: {var_name}")
            if has_time:
                print(f"   时间范围: 1980-1993")
                print(f"   时间步数: {len(var_full.time)}")
            else:
                print(f"   静态场 (无时间维度)")
            print(f"   总层数: {n_levels}")
            print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
            
        else:
            # 2D数据处理
            if has_time:
                var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(time=slice("1980", "1993"))
                print(f"✅ 2D数据加载完成")
                print(f"   时间步数: {len(var_full.time)}")
            else:
                var_full = cat.ICON.C5[dataset_key].to_dask()[var_name]
                print(f"✅ 静态2D数据加载完成")
        
        print("="*70)
        
        # 初始化失败层级列表
        failed_levels = []
        
        if has_level:
            # 3D数据：逐层处理
            total_start_time = time.time()
            
            for idx, level in enumerate(levels, 1):
                level_start_time = time.time()
                
                # 检查内存状态
                mem_ok = print_memory_status(f"[Layer {idx}/{n_levels}]")
                mem_status = get_memory_usage()
                
                if mem_status['system_percent'] > memory_threshold:
                    print(f"⚠️ 内存使用率过高 ({mem_status['system_percent']:.1f}% > {memory_threshold}%)")
                    print(f"   跳过 Level {int(level):3d}，稍后重试")
                    failed_levels.append(int(level))
                    
                    # 强制垃圾回收
                    gc.collect()
                    time.sleep(2)  # 等待内存释放
                    continue
                
                # 构建保存路径
                save_path = os.path.join(exp_save_dir, f"{var_name}_lev_{int(level):03d}.nc")
                
                # 检查是否已处理
                if os.path.exists(save_path):
                    print(f"✅ [{idx}/{n_levels}] Level {int(level):3d} - 已存在，跳过")
                    continue
                
                print(f"🔄 [{idx}/{n_levels}] 处理 Level {int(level):3d}...")
                
                # 重试机制
                success = False
                for attempt in range(max_retries):
                    try:
                        # 1. 选择单层
                        layer_data = var_full.sel({level_dim_name: level})
                        print(f"   ├─ [尝试 {attempt+1}/{max_retries}] 选择层级完成")
                        
                        # 2. 转换到经纬度网格
                        layer_lonlat = dataarray_to_equatorial_latlon_grid(
                            layer_data, 'healpix', grid_dict
                        )
                        print(f"   ├─ 网格转换完成: {layer_lonlat.shape}")
                        
                        # 3. 插值到2°x2°
                        layer_2deg = layer_lonlat.interp(
                            lat=target_lat, lon=target_lon, method='linear'
                        )
                        print(f"   ├─ 插值完成: {layer_2deg.shape}")
                        
                        # 4. 计算并保存（小批量处理）
                        print(f"   ├─ 计算并保存中...")
                        
                        # 使用 ProgressBar 监控
                        with ProgressBar():
                            layer_computed = layer_2deg.compute()
                        
                        # 保存
                        ds_to_save = layer_computed.to_dataset(name=var_name)
                        ds_to_save.to_netcdf(save_path)
                        
                        layer_time = time.time() - level_start_time
                        print(f"   ✅ Level {int(level):3d} 完成 (耗时: {layer_time:.1f}s)")
                        
                        # 清理内存
                        del layer_data, layer_lonlat, layer_2deg, layer_computed, ds_to_save
                        gc.collect()
                        
                        success = True
                        break  # 成功则跳出重试循环
                        
                    except MemoryError as e:
                        print(f"   ⚠️ [尝试 {attempt+1}/{max_retries}] 内存错误: {str(e)}")
                        gc.collect()
                        time.sleep(5)  # 等待内存释放
                        if attempt == max_retries - 1:
                            print(f"   ❌ Level {int(level):3d} 处理失败（内存不足）")
                            failed_levels.append(int(level))
                    
                    except Exception as e:
                        print(f"   ⚠️ [尝试 {attempt+1}/{max_retries}] 错误: {str(e)}")
                        gc.collect()
                        time.sleep(2)
                        if attempt == max_retries - 1:
                            print(f"   ❌ Level {int(level):3d} 处理失败")
                            failed_levels.append(int(level))
                
                # 更新进度
                with open(progress_file, 'w') as f:
                    f.write(f"Last processed: Level {int(level):03d} ({idx}/{n_levels})\n")
                    f.write(f"Time: {time.ctime()}\n")
            
            # 保存失败的层级
            if failed_levels:
                with open(failed_levels_file, 'w') as f:
                    f.write("Failed levels:\n")
                    for lev in failed_levels:
                        f.write(f"{lev}\n")
                print(f"\n⚠️ 有 {len(failed_levels)} 层处理失败，已记录到: {failed_levels_file}")
            
            total_time = time.time() - total_start_time
            print(f"\n🎉 {var_name.upper()} - {experiment_name} 处理完成!")
            print(f"⏱️  总耗时: {total_time/60:.1f} 分钟")
            print(f"✅ 成功: {n_levels - len(failed_levels)}/{n_levels} 层")
            
        else:
            # 2D数据处理
            print("🔄 处理2D数据...")
            save_path = os.path.join(exp_save_dir, f"{var_name}_2deg.nc")
            
            if os.path.exists(save_path):
                print(f"✅ 文件已存在，跳过: {save_path}")
            else:
                try:
                    # 转换网格
                    var_lonlat = dataarray_to_equatorial_latlon_grid(var_full, 'healpix', grid_dict)
                    print(f"✅ 网格转换完成")
                    
                    # 插值
                    var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                    print(f"✅ 插值完成")
                    
                    # 计算并保存
                    print(f"🔄 计算并保存...")
                    with ProgressBar():
                        var_computed = var_2deg.compute()
                    
                    ds_to_save = var_computed.to_dataset(name=var_name)
                    ds_to_save.to_netcdf(save_path)
                    print(f"✅ 保存完成: {save_path}")
                    
                    del var_lonlat, var_2deg, var_computed, ds_to_save
                    gc.collect()
                    
                except Exception as e:
                    print(f"❌ 2D数据处理失败: {str(e)}")
                    raise
        
        print("="*70)
        print_memory_status("(处理完成后)")
        
    except Exception as e:
        print(f"❌ 数据加载失败: {str(e)}")
        import traceback
        traceback.print_exc()
        gc.collect()
        raise
    
    # 最终清理
    gc.collect()
    
    return failed_levels if has_level else []

print("✅ 增强版处理函数已定义: process_var_data_safe()")

✅ 增强版处理函数已定义: process_var_data_safe()


In [30]:
def process_zg_static_field(experiment_name, dataset_key, save_dir, grid_dict, target_lat, target_lon, level_slice=(0, None)):
    """
    专门处理 zg 静态场变量（无时间维度）
    
    Parameters:
    -----------
    experiment_name : str
        实验名称 (CNTL, P4K, 4CO2)
    dataset_key : str
        在catalog中的数据集键名
    save_dir : str
        保存目录
    grid_dict : dict
        网格转换参数
    target_lat : array
        目标纬度
    target_lon : array
        目标经度
    level_slice : tuple
        level切片范围 (start, end)
    """
    import time
    import gc
    
    var_name = 'zg'
    
    print("="*70)
    print(f"🔄 处理静态场变量: ZG - {experiment_name}")
    print("="*70)
    
    # 创建保存目录
    exp_save_dir = os.path.join(save_dir, f"zg_{experiment_name.lower()}_layers")
    os.makedirs(exp_save_dir, exist_ok=True)
    print(f"📁 保存路径: {exp_save_dir}")
    
    # 加载 zg 数据
    print(f"📖 读取 ZG 静态场数据...")
    
    try:
        # zg 没有时间维度，直接加载
        zg_data = cat.ICON.C5[dataset_key].to_dask()['zg']
        
        # 检查维度
        print(f"✅ 数据信息:")
        print(f"   变量: zg (地球势高)")
        print(f"   维度: {list(zg_data.dims)}")
        print(f"   形状: {zg_data.shape}")
        
        # 检测 level 维度
        level_dim_name = None
        possible_level_dims = ['level_full', 'level_half', 'level']
        for dim in possible_level_dims:
            if dim in zg_data.dims:
                level_dim_name = dim
                break
        
        if level_dim_name is None:
            raise ValueError(f"无法找到 level 维度。zg 的维度: {list(zg_data.dims)}")
        
        print(f"✅ 检测到 level 维度: {level_dim_name}")
        
        # 选择 level 范围
        zg_selected = zg_data.sel({level_dim_name: slice(*level_slice)})
        levels = zg_selected[level_dim_name].values
        n_levels = len(levels)
        
        print(f"   总层数: {n_levels}")
        print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
        print("="*70)
        
        # 逐层处理
        total_start_time = time.time()
        
        for idx, level in enumerate(levels, 1):
            # 构建保存路径
            save_path = os.path.join(exp_save_dir, f"zg_lev_{int(level):03d}.nc")
            
            # 检查是否已处理
            if os.path.exists(save_path):
                print(f"✅ [{idx}/{n_levels}] Level {int(level):3d} - 已存在，跳过")
                continue
            
            print(f"🔄 [{idx}/{n_levels}] 处理 Level {int(level):3d}...")
            
            try:
                # 1. 选择单层数据
                zg_layer = zg_selected.sel({level_dim_name: level})
                print(f"   ├─ 选择层级完成")
                
                # 2. 转换到经纬度网格
                zg_lonlat = dataarray_to_equatorial_latlon_grid(zg_layer, 'healpix', grid_dict)
                print(f"   ├─ 网格转换完成: {zg_lonlat.shape}")
                
                # 3. 插值到2°x2°
                zg_2deg = zg_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                print(f"   ├─ 插值完成: {zg_2deg.shape}")
                
                # 4. 计算并保存
                print(f"   ├─ 计算并保存中...")
                with ProgressBar():
                    zg_2deg_computed = zg_2deg.compute()
                
                ds_to_save = zg_2deg_computed.to_dataset(name='zg')
                ds_to_save.to_netcdf(save_path)
                
                print(f"   ✅ Level {int(level):3d} 完成")
                
                # 清理内存
                del zg_layer, zg_lonlat, zg_2deg, zg_2deg_computed, ds_to_save
                gc.collect()
                
            except MemoryError:
                print(f"   ❌ 内存不足，跳过此层")
                gc.collect()
                continue
            except Exception as e:
                print(f"   ❌ 处理失败: {str(e)}")
                gc.collect()
                continue
        
        total_time = time.time() - total_start_time
        print(f"\n🎉 ZG - {experiment_name} 处理完成!")
        print(f"⏱️  总耗时: {total_time/60:.1f} 分钟")
        print("="*70)
        
    except Exception as e:
        print(f"❌ ZG 数据加载失败: {str(e)}")
        gc.collect()
        raise
    
    # 最后清理
    gc.collect()


## 循环转换数据-气压层

In [31]:


# 网格参数
grid_dict = {"nside": 256, "nest": True, "minmax_lat": 36}
target_lat = np.arange(-36, 36.1, 2.0)
target_lon = np.arange(0, 360, 2.0)

# 要处理的变量列表（3D和2D）
variables_3d = [
    # 'phalf'
    # 'rho'
    # "ua", "va"
    # "pfull"
    "wa",
    # "zg"  # zg 单独处理，不放在这里
    ]  # 3D变量：需要逐层处理

# zg 变量（静态场，无时间维度）
variables_zg = ["cell_sea_land_mask"]  # 单独处理

variables_2d = [
                # "pr", 
                # "ta", 
                # "hus", "ua", "va",
                # "hfls", "hfss", 
                # "rsdt", "rsut", "rlut",   
                # "rsds", "rsus", "rlds", "rlus",
                # "sfcwind",
                # "ts","tas",
                # "hus2m"
                # "ps"
              
                ]
          
if variables_2d:
    # 设置保存目录
    LAYER_DIR = os.path.join(DATA_DIR, "2d_layers")
    os.makedirs(LAYER_DIR, exist_ok=True)
else:
    LAYER_DIR = os.path.join(DATA_DIR, "3d_layers")
    os.makedirs(LAYER_DIR, exist_ok=True)   



# 定义实验配置
experiments = {
    "cntl":  ("CNTL",   "AMIP_CNTL"),
    "4k":    ("P4K",    "AMIP_P4K"),
    # "4co2":  ("4CO2",   "AMIP_4CO2"),
}


# 记录失败的任务
failed_tasks = []

# ==================== 1. 处理 ZG 静态场变量 ====================
# if variables_zg:
#     print("\n" + "🌍"*35)
#     print("📊 开始处理静态场变量: ZG (地球势高)")
#     print("🌍"*35 + "\n")
    
#     for exp_key, (exp_name, dataset_key) in experiments.items():
#         try:
#             process_zg_static_field(
#                 experiment_name=exp_name,
#                 dataset_key=dataset_key,
#                 save_dir=LAYER_DIR,
#                 grid_dict=grid_dict,
#                 target_lat=target_lat,
#                 target_lon=target_lon,
#                 level_slice=(0, None)  # 处理所有层
#             )
#         except MemoryError as e:
#             error_msg = f"ZG - {exp_name}: 内存不足"
#             print(f"❌ {error_msg}")
#             failed_tasks.append(error_msg)
#             gc.collect()
#             time.sleep(10)
#             continue
#         except Exception as e:
#             error_msg = f"ZG - {exp_name}: {str(e)}"
#             print(f"❌ {error_msg}")
#             failed_tasks.append(error_msg)
#             gc.collect()
#             continue

# ==================== 2. 处理3D变量（有时间维度） ====================
# # 处理3D变量
# for var_name in variables_3d:
#     print("\n" + "="*70)
#     print(f"📊 开始处理3D变量: {var_name.upper()}")
#     print("="*70 + "\n")
    
#     for exp_key, (exp_name, dataset_key) in experiments.items():
#         try:
      
            
#             process_var_data(
#                 var_name=var_name,
#                 experiment_name=exp_name,
#                 dataset_key=dataset_key,
#                 save_dir=LAYER_DIR,
#                 grid_dict=grid_dict,
#                 target_lat=target_lat,
#                 target_lon=target_lon,
#                 has_level=True,  # 3D数据
#                 has_time=True  # 3D变量都有时间维度
#             )
#         except MemoryError as e:
#             error_msg = f"{var_name.upper()} - {exp_name}: 内存不足"
#             print(f"❌ {error_msg}")
#             failed_tasks.append(error_msg)
#             gc.collect()
           
#             continue
#         except Exception as e:
#             error_msg = f"{var_name.upper()} - {exp_name}: {str(e)}"
#             print(f"❌ {error_msg}")
#             failed_tasks.append(error_msg)
#             gc.collect()
#             continue

# ==================== 3. 处理2D变量（时间序列） ====================
# 处理2D变量
# for var_name in variables_2d:
#     print("\n" + "="*70)
#     print(f"📊 开始处理2D变量: {var_name.upper()}")
#     print("="*70 + "\n")
    
#     for exp_key, (exp_name, dataset_key) in experiments.items():
#         try:
#             # 处理前检查内存
     
            
#             process_var_data(
#                 var_name=var_name,
#                 experiment_name=exp_name,
#                 dataset_key=dataset_key,
#                 save_dir=LAYER_DIR,
#                 grid_dict=grid_dict,
#                 target_lat=target_lat,
#                 target_lon=target_lon,
#                 has_level=False,  # 2D数据
#                 has_time=True  # 2D数据默认都有时间维度
#             )
#         except MemoryError as e:
#             error_msg = f"{var_name.upper()} - {exp_name}: 内存不足"
#             print(f"❌ {error_msg}")
#             failed_tasks.append(error_msg)
#             gc.collect()
#             time.sleep(10)
#             continue
#         except Exception as e:
#             error_msg = f"{var_name.upper()} - {exp_name}: {str(e)}"
#             print(f"❌ {error_msg}")
#             failed_tasks.append(error_msg)
#             gc.collect()
#             continue



# # 显示失败任务
# if failed_tasks:
#     print("\n" + "="*70)
#     print("⚠️ 以下任务处理失败:")
#     print("="*70)
#     for task in failed_tasks:
#         print(f"  - {task}")
# else:
#     print("\n✅ 所有任务成功完成！")



## 批量处理多个实验的海陆Mask

In [32]:
# 批量处理所有实验的海陆mask
from process_sea_land_mask import batch_process_sea_land_mask

results = batch_process_sea_land_mask(
    experiments=experiments,
    save_dir=LAYER_DIR,
    grid_dict=grid_dict,
    target_lat=target_lat,
    target_lon=target_lon,
    catalog=cat
)

print(f"\n✅ 批处理完成！")
print(f"已处理实验: {list(results.keys())}")


🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍
📊 批量处理海陆Mask变量
🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍

🌊 处理海陆Mask变量: CELL_SEA_LAND_MASK - CNTL
📁 保存路径: /work/mh1498/m301257/processed_data_lat_30/3d_layers/mask_cntl
✅ 文件已存在，跳过处理
   文件: /work/mh1498/m301257/processed_data_lat_30/3d_layers/mask_cntl/cell_sea_land_mask_2deg.nc
✅ CNTL 处理成功

🌊 处理海陆Mask变量: CELL_SEA_LAND_MASK - P4K
📁 保存路径: /work/mh1498/m301257/processed_data_lat_30/3d_layers/mask_p4k
✅ 文件已存在，跳过处理
   文件: /work/mh1498/m301257/processed_data_lat_30/3d_layers/mask_p4k/cell_sea_land_mask_2deg.nc
✅ P4K 处理成功


📊 处理总结
✅ 成功: 2 个实验
   - CNTL: /work/mh1498/m301257/processed_data_lat_30/3d_layers/mask_cntl/cell_sea_land_mask_2deg.nc
   - P4K: /work/mh1498/m301257/processed_data_lat_30/3d_layers/mask_p4k/cell_sea_land_mask_2deg.nc

✅ 批处理完成！
已处理实验: ['CNTL', 'P4K']


In [33]:
# 验证生成的文件
import xarray as xr

mask_file = "/work/mh1498/m301257/processed_data_lat_30/2d_layers/mask_cntl/cell_sea_land_mask_2deg.nc"
ds = xr.open_dataset(mask_file)

print("📊 文件验证结果：")
print(f"变量: {list(ds.data_vars)}")
print(f"维度: {list(ds.dims)}")
print(f"形状: {ds['cell_sea_land_mask'].shape}")
print(f"\n数据属性:")
for key, value in ds['cell_sea_land_mask'].attrs.items():
    print(f"  {key}: {value}")

ds.close()

📊 文件验证结果：
变量: ['cell_sea_land_mask']
维度: ['lat', 'lon']
形状: (37, 180)

数据属性:
  long_name: Sea-Land Mask
  description: Static sea-land mask: 0=land, 1=sea
  source: ICON CNTL experiment
  grid_resolution: 2 degrees
  original_grid: HEALPix
  interpolation_method: nearest neighbor


## 批量处理所有实验的wa变量

In [36]:
# 批量处理所有实验
from process_3d_data_optimized import batch_process_3d_variables

# 定义要处理的变量
variables_to_process = ["wa"]  # 可以添加更多: ["wa", "ua", "va", "hus"]

# 批量处理
all_results = batch_process_3d_variables(
    var_names=variables_to_process,
    experiments=experiments,
    save_dir=LAYER_DIR,
    grid_dict=grid_dict,
    target_lat=target_lat,
    target_lon=target_lon,
    catalog=cat,
    time_batch_size=730,  # 每次处理2年
    memory_threshold=85,
    max_retries=3,
    skip_existing=True
)

print("\n🎉 批量处理完成！")


🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍
📊 批量处理3D变量
🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍🌍


📊 开始处理变量: WA

🔄 处理3D变量: WA - CNTL
💾 内存 (开始): 进程 16.50GB | 系统 325.1/501.9GB (35.2%)
📁 保存路径: /work/mh1498/m301257/processed_data_lat_30/3d_layers/wa_cntl_layers
📖 读取数据元信息...
✅ 检测到level维度: level_full
✅ 数据信息:
   变量: wa
   时间范围: 1980-1993
   时间步数: 5114 天
   总层数: 26
   层级范围: 14.0 - 90.0
   时间批次大小: 730 天
   时间批次数: 8
✅ [1/26] Level  14 - 已存在，跳过
✅ [2/26] Level  21 - 已存在，跳过
✅ [3/26] Level  25 - 已存在，跳过
✅ [4/26] Level  29 - 已存在，跳过
✅ [5/26] Level  31 - 已存在，跳过
✅ [6/26] Level  35 - 已存在，跳过
✅ [7/26] Level  38 - 已存在，跳过
✅ [8/26] Level  41 - 已存在，跳过
✅ [9/26] Level  46 - 已存在，跳过
✅ [10/26] Level  51 - 已存在，跳过
✅ [11/26] Level  55 - 已存在，跳过
✅ [12/26] Level  58 - 已存在，跳过
✅ [13/26] Level  63 - 已存在，跳过
✅ [14/26] Level  67 - 已存在，跳过
✅ [15/26] Level  71 - 已存在，跳过
✅ [16/26] Level  74 - 已存在，跳过
✅ [17/26] Level  76 - 已存在，跳过
✅ [18/26] Level  78 - 已存在，跳过
✅ [19/26] Level  80 - 已存在，跳过
✅ [20/26] Level  81 - 已存在，跳过
✅ [21/26] Level  83 - 已存在，跳过
✅ [

/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


✅ 检测到level维度: level_full
✅ 数据信息:
   变量: wa
   时间范围: 1980-1993
   时间步数: 5114 天
   总层数: 26
   层级范围: 14.0 - 90.0
   时间批次大小: 730 天
   时间批次数: 8
✅ [1/26] Level  14 - 已存在，跳过
✅ [2/26] Level  21 - 已存在，跳过
✅ [3/26] Level  25 - 已存在，跳过
✅ [4/26] Level  29 - 已存在，跳过
✅ [5/26] Level  31 - 已存在，跳过
✅ [6/26] Level  35 - 已存在，跳过
✅ [7/26] Level  38 - 已存在，跳过
✅ [8/26] Level  41 - 已存在，跳过
✅ [9/26] Level  46 - 已存在，跳过
✅ [10/26] Level  51 - 已存在，跳过
✅ [11/26] Level  55 - 已存在，跳过
✅ [12/26] Level  58 - 已存在，跳过
✅ [13/26] Level  63 - 已存在，跳过
✅ [14/26] Level  67 - 已存在，跳过
✅ [15/26] Level  71 - 已存在，跳过
✅ [16/26] Level  74 - 已存在，跳过
✅ [17/26] Level  76 - 已存在，跳过
✅ [18/26] Level  78 - 已存在，跳过
✅ [19/26] Level  80 - 已存在，跳过
✅ [20/26] Level  81 - 已存在，跳过
✅ [21/26] Level  83 - 已存在，跳过
✅ [22/26] Level  84 - 已存在，跳过
✅ [23/26] Level  85 - 已存在，跳过
✅ [24/26] Level  87 - 已存在，跳过
✅ [25/26] Level  89 - 已存在，跳过
✅ [26/26] Level  90 - 已存在，跳过

📊 处理总结: WA - P4K
✅ 成功处理: 0 层
⏭️  跳过（已存在）: 26 层
❌ 处理失败: 0 层
⏱️  总耗时: 0.1 分钟
📁 保存目录: /work/mh1498/m301257/processed_d

In [37]:
# 查看批量处理结果总结
print("\n" + "="*70)
print("📊 批量处理结果总结")
print("="*70)

for var_name, var_results in all_results.items():
    print(f"\n{var_name.upper()}:")
    for exp_name, result in var_results.items():
        if 'error' in result:
            print(f"  ❌ {exp_name}: 处理失败")
        else:
            print(f"  ✅ {exp_name}:")
            print(f"     - 成功处理: {result['success']} 层")
            print(f"     - 跳过（已存在）: {result['skipped']} 层")
            print(f"     - 失败: {result['failed']} 层")
            print(f"     - 总耗时: {result['total_time']/60:.1f} 分钟")
            
            # 检查文件
            exp_dir = f"{LAYER_DIR}/{var_name}_{exp_name.lower()}_layers"
            if os.path.exists(exp_dir):
                files = glob.glob(f"{exp_dir}/{var_name}_lev_*.nc")
                total_size = sum(os.path.getsize(f) for f in files) / 1024**3
                print(f"     - 文件数: {len(files)} 个")
                print(f"     - 总大小: {total_size:.2f} GB")


📊 批量处理结果总结

WA:
  ✅ CNTL:
     - 成功处理: 0 层
     - 跳过（已存在）: 26 层
     - 失败: 0 层
     - 总耗时: 0.0 分钟
     - 文件数: 26 个
     - 总大小: 6.60 GB
  ✅ P4K:
     - 成功处理: 0 层
     - 跳过（已存在）: 26 层
     - 失败: 0 层
     - 总耗时: 0.1 分钟
     - 文件数: 26 个
     - 总大小: 6.60 GB


In [38]:
def change_variable_name_and_merge_per_folder(in_paths, base_out_path, pattern,
                                              old_var_name, new_var_name,
                                              merged_file_name="wa_all_levels.nc",
                                              skip_existing=True,
                                              has_level=True):
    """
    批量更改NetCDF文件中的变量名，添加level维度（如果需要），并在每个子文件夹生成合并文件
    同时保留输入子文件夹结构
    
    Parameters:
    -----------
    skip_existing : bool
        如果为True，跳过已存在的单个文件和合并文件（默认True）
    has_level : bool
        如果为True，处理3D数据（从文件名提取level维度）
        如果为False，处理2D数据（直接合并，不添加level维度）
    """
    import os
    import glob
    import xarray as xr

    for in_path in in_paths:
        # 当前输入文件夹名称
        folder_name = os.path.basename(os.path.normpath(in_path))
        out_path = os.path.join(base_out_path, folder_name)
        os.makedirs(out_path, exist_ok=True)
        
        # 检查合并文件是否已存在
        merged_file = os.path.join(out_path, merged_file_name)
        if skip_existing and os.path.exists(merged_file):
            print(f"✅ {folder_name}: 合并文件已存在，跳过处理")
            print(f"   文件: {merged_file}")
            continue

        file_pattern = os.path.join(in_path, pattern)
        input_files = sorted(glob.glob(file_pattern))
        
        if not input_files:
            print(f"⚠️ 文件夹 {folder_name} 没有匹配的文件 (pattern: {pattern})")
            continue
        
        data_type = "3D (多层级)" if has_level else "2D (单层/时间序列)"
        print(f"\n{'='*70}")
        print(f"🔄 处理文件夹: {folder_name} [{data_type}]")
        print(f"   输入路径: {in_path}")
        print(f"   输出路径: {out_path}")
        print(f"   找到文件数: {len(input_files)}")
        print(f"{'='*70}")
        
        datasets = []
        processed_count = 0
        skipped_count = 0

        for file in input_files:
            filename = os.path.basename(file)
            new_file = os.path.join(out_path, filename)

            # 检查单个文件是否已存在
            if skip_existing and os.path.exists(new_file):
                skipped_count += 1
                # 仍需加载用于合并
                try:
                    with xr.open_dataset(new_file) as ds:
                        if has_level:
                            # 3D数据：需要level维度
                            level = int(filename.split("_")[2].replace(".nc", ""))
                            ds = ds.expand_dims({"level": [level]}) if "level" not in ds.dims else ds
                        datasets.append(ds)
                except Exception as e:
                    print(f"⚠️ 跳过的文件加载失败: {filename} - {str(e)}")
                continue

            # 处理新文件
            try:
                if has_level:
                    # 3D数据：从文件名提取 level (例如: hus_lev_031.nc -> 31)
                    try:
                        level = int(filename.split("_")[2].replace(".nc", ""))
                    except:
                        print(f"⚠️ 无法从文件名提取 level: {filename}, 跳过")
                        continue
                
                with xr.open_dataset(file) as ds:
                    # 修改变量名（如果需要）
                    if old_var_name in ds and old_var_name != new_var_name:
                        ds = ds.rename({old_var_name: new_var_name})
                    
                    # 3D数据需要添加 level 维度
                    if has_level:
                        ds = ds.expand_dims({"level": [level]})
                    
                    # 保存到新路径
                    ds.to_netcdf(new_file, mode="w")
                    processed_count += 1
                    
                    if processed_count % 5 == 0 or processed_count == len(input_files):
                        print(f"✅ [{processed_count}/{len(input_files)}] 已处理: {filename}")
                    
                    datasets.append(ds)
            except Exception as e:
                print(f"❌ 处理失败: {filename} - {str(e)}")
                continue

        # 统计信息
        print(f"\n📊 处理统计:")
        print(f"   新处理: {processed_count} 个文件")
        print(f"   已跳过: {skipped_count} 个文件")
        print(f"   总计: {len(datasets)} 个文件用于合并")

        # 每个子文件夹单独合并
        if datasets:
            try:
                if has_level:
                    # 3D数据：沿level维度合并
                    ds_all = xr.concat(datasets, dim="level")
                else:
                    # 2D数据：直接合并（沿时间或其他维度）
                    ds_all = xr.concat(datasets, dim="time") if "time" in datasets[0].dims else xr.merge(datasets)
                
                var_data = ds_all[new_var_name]
                var_data.to_netcdf(merged_file)
                print(f"🎉 {folder_name} 合并完成!")
                print(f"   保存到: {merged_file}")
                print(f"   形状: {var_data.shape}")
                print(f"   维度: {list(var_data.dims)}")
            except Exception as e:
                print(f"❌ 合并失败: {str(e)}")
        else:
            print(f"⚠️ 文件夹 {folder_name} 没有可处理的文件！")


## merge_files

In [39]:
# input_folders1 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_p4k_layers",

# ]

# input_folders2 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/va_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/va_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/va_p4k_layers"
# ]

# input_folders3 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_p4k_layers"
# ]
# base_output_folder = "/work/mh1498/m301257"

# # 处理3D数据 (hus - 比湿)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders1,
#     base_out_path=base_output_folder,
#     pattern="ua_lev_*.nc",
#     old_var_name="ua",
#     new_var_name="ua",
#     has_level=True,  # 3D数据，有level维度,
#     merged_file_name="ua_all_levels.nc"
# )

# # 处理3D数据 (ta - 温度)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders2,
#     base_out_path=base_output_folder,
#     pattern="va_lev_*.nc",
#     old_var_name="va",
#     new_var_name="va",
#     has_level=True , # 3D数据，有level维度
#     merged_file_name="va_all_levels.nc"
# )


# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders3,
#     base_out_path=base_output_folder,
#     pattern="pfull_lev_*.nc",
#     old_var_name="pfull",
#     new_var_name="pfull",
#     has_level=True , # 3D数据，有level维度
#     merged_file_name="pfull_all_levels.nc"
# )

## 测试区域（可选）

In [40]:
# 测试 ZG 处理（只处理前3层）
# test_experiments = {"cntl": ("CNTL", "AMIP_CNTL")}
# try:
#     process_zg_static_field(
#         experiment_name="CNTL",
#         dataset_key="AMIP_CNTL", 
#         save_dir=LAYER_DIR,
#         grid_dict=grid_dict,
#         target_lat=target_lat,
#         target_lon=target_lon,
#         level_slice=(0, 3)  # 只处理前3层
#     )
#     print("✅ ZG 测试成功！")
# except Exception as e:
#     print(f"❌ 测试失败: {e}")
#     import traceback
#     traceback.print_exc()
